# Racial Equity in U.S. Grantmaking: Findings

**Research Question:** How has funding for racial equity in the United States changed over time, and what patterns exist in who gives, who receives, and where the money flows?

**Data Sources:**
- IRS 990-PF e-file XML filings (public S3 bucket, 2011–2023)
- ProPublica Nonprofit Explorer API (org profiles, NTEE codes)
- Candid Demographics API *(pending — to be enriched after API access)*

**Methodology:** We classify grants as racial-equity-related using a keyword regex applied to the `grant_purpose` field, following the general approach described by [Candid and the Philanthropic Initiative for Racial Equity (PRE)](https://blog.candid.org/post/what-counts-as-racial-equity-funding/). Keyword list is in `src/data_cleaning.py`.

---

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

from src.data_cleaning import get_connection

%matplotlib inline
plt.rcParams['figure.dpi'] = 130

DB   = Path('../data/racial_equity_grants.sqlite')
PROC = Path('../data/processed')
conn = get_connection(DB)

grants_df = pd.read_sql('SELECT * FROM grants', conn)
re_grants  = grants_df[grants_df['is_racial_equity'] == 1]

## Key Statistics

> *Update these cells after running Notebooks 01–03.*

In [ ]:
s = pd.read_sql("""
    SELECT COUNT(*) AS total_grants,
           COUNT(DISTINCT funder_ein) AS unique_funders,
           COUNT(DISTINCT recipient_ein) AS unique_recipients,
           ROUND(SUM(grant_amount)/1e6, 1) AS total_dollars_m,
           ROUND(AVG(grant_amount)) AS avg_grant_size,
           MIN(tax_year) AS earliest_year,
           MAX(tax_year) AS latest_year
    FROM grants WHERE is_racial_equity = 1
""", conn).iloc[0]

print(f"""
Dataset covers: {s.earliest_year}–{s.latest_year}
Racial equity grants identified: {s.total_grants:,}
Total dollars: ${s.total_dollars_m:,.0f}M
Average grant size: ${s.avg_grant_size:,.0f}
Unique funders: {s.unique_funders:,}
Unique recipients: {s.unique_recipients:,}
""")

## Finding 1: The 2020 Surge

Racial equity grantmaking rose sharply following the murder of George Floyd and the
2020 racial justice uprisings. The chart below shows annual grant totals; the red dashed
line marks 2020.

In [ ]:
img_path = PROC / 'fig_funding_over_time.png'
if img_path.exists():
    plt.figure(figsize=(12, 5))
    plt.imshow(mpimg.imread(img_path))
    plt.axis('off')
    plt.show()
else:
    print("Run Notebook 03 first to generate this figure.")

In [ ]:
# Pre/post 2020 numbers inline
period = pd.read_sql("""
    SELECT CASE WHEN tax_year >= 2020 THEN 'Post-2020' ELSE 'Pre-2020' END AS period,
           COUNT(DISTINCT tax_year) AS years,
           ROUND(SUM(grant_amount)/1e6, 1) AS total_m,
           ROUND(SUM(grant_amount)/COUNT(DISTINCT tax_year)/1e6, 1) AS avg_annual_m
    FROM grants
    WHERE is_racial_equity = 1 AND tax_year IS NOT NULL
    GROUP BY period
""", conn)
period

## Finding 2: Funder Concentration

Racial equity philanthropy is concentrated: a small number of large foundations account for
the majority of dollars. The top 20 funders by total racial equity giving are shown below.

In [ ]:
img_path = PROC / 'fig_top_funders.png'
if img_path.exists():
    plt.figure(figsize=(10, 8))
    plt.imshow(mpimg.imread(img_path))
    plt.axis('off')
    plt.show()

## Finding 3: Geographic Concentration

Racial equity grant dollars flow disproportionately to organizations in a handful of states.
New York, California, and the District of Columbia consistently rank at the top — reflecting
both population size and the concentration of large nonprofit organizations.

In [ ]:
pd.read_sql("""
    SELECT recipient_state,
           COUNT(*) AS grants,
           ROUND(SUM(grant_amount)/1e6, 1) AS dollars_m
    FROM grants
    WHERE is_racial_equity=1 AND recipient_state IS NOT NULL
    GROUP BY recipient_state
    ORDER BY SUM(grant_amount) DESC
    LIMIT 10
""", conn)

## Finding 4: Issue Area Breakdown

Racial equity grants span a wide range of issue areas. Civil rights / advocacy organizations
receive the largest share, followed by community development, education, and health.

In [ ]:
img_path = PROC / 'fig_ntee.png'
if img_path.exists():
    plt.figure(figsize=(9, 6))
    plt.imshow(mpimg.imread(img_path))
    plt.axis('off')
    plt.show()
else:
    print("NTEE data requires ProPublica enrichment — run Notebook 01 first.")

## Methodology Notes

- **Keyword classification** is a lower bound: grants with vague purpose text are not captured. Candid's own methodology uses a combination of grant descriptions, grantee mission statements, and Population codes — this project replicates only the description-based component.
- **990-PF sampling** means smaller foundations that filed only paper returns are not included.
- **Pre/post significance testing** uses Welch's t-test; the small number of post-2020 years (≤ 4) limits statistical power.
- After Candid API access is granted, the `demographics` table will be populated to analyze leadership diversity of funded organizations.

## Limitations & Future Directions

1. Add Candid Premier API grants data for richer coverage (corporate foundations, donor-advised funds).
2. Apply Candid's Population codes to improve racial equity classification accuracy.
3. Normalize grant dollars to per-capita by state population using Census data.
4. Longitudinal analysis of individual funder commitment — did post-2020 surge persist?

---
*Built with [Claude Code](https://claude.ai/claude-code). Data: IRS 990-PF e-file (public domain), ProPublica Nonprofit Explorer API.*